In [0]:


bronze_path = "abfss://bronze-layer@stockdatalake1.dfs.core.windows.net/stock_raw.csv"
df = spark.read.format("csv").options(**configs).load(bronze_path)

In [0]:
# Read CSV with header and schema inference
df = spark.read.format("csv") \
    .options(**configs) \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path)

# Show the first few row

In [0]:
df.display()

In [0]:
df_silver = df.toDF(*[c.strip() for c in df.columns])

In [0]:
from pyspark.sql import functions as F

df_silver = df_silver \
    .withColumn("Date", F.try_to_date(F.col("Date"), "dd-MMM-yyyy")) \
    .withColumn("Open", F.col("Open").cast("double")) \
    .withColumn("High", F.col("High").cast("double")) \
    .withColumn("Low", F.col("Low").cast("double")) \
    .withColumn("Close", F.col("Close").cast("double")) \
    .withColumn("shares_traded", F.col("Shares Traded").cast("int")) \
    .withColumn("Turnover_cr", F.col("Turnover (₹ Cr)").cast("double"))

# Check bad records (important in production)
df_silver.filter(F.col("Date").isNull()).show()

df_silver.printSchema()
df_silver.show(5)

In [0]:
df_silver.withColumnRenamed("Date","date").withColumnRenamed("Open","open").withColumnRenamed("High","high").withColumnRenamed("Low","low").withColumnRenamed("Close","close").withColumnRenamed("Volume","volume").withColumnRenamed("Turnover (₹ Cr)","turnover").withColumnRenamed("Shares Traded","shares_traded").display()

In [0]:
df_silver.display()

In [0]:
# Suppose df_bronze has a column "unnecessary_column"
df_silver = df_silver.drop("Shares Traded" ,"Turnover (₹ Cr)" )


In [0]:
df_silver.display()

In [0]:
bronze_path = "abfss://bronze-layer@stockdatalake1.dfs.core.windows.net/stock_raw.csv"
silver_path = "abfss://silver-layer@stockdatalake1.dfs.core.windows.net/stock"

# Set OAuth configs per storage account
for k, v in configs.items():
    spark.conf.set(k, v)

# Read CSV with header and schema
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path)

# Transform data if needed
df_silver = df  # example

# Write Delta table
df_silver.write.mode("overwrite").format("delta").save(silver_path)

In [0]:
df_silver.write.mode("overWrite").format("delta").save("abfss://silver-layer@stockdatalake1.dfs.core.windows.net/stock")